# **ETL de datos**

Fuentes:
- https://datos.sonora.gob.mx/dataset/Fiebre%20manchada%20%28Rickettsia%29
- https://smn.conagua.gob.mx/es/climatologia/informacion-climatologica/normales-climatologicas-por-estado?estado=son


**Proceso:**

0. Instalación de librerías y dependencias
1. Lectura de datos desde fuente para cada año
2. Concatenación de datos en un solo histórico
3. Guardado de datos en /data/processed

## 0. Instalación de librerías y dependencias

In [1]:
import pandas as pd
import requests
from io import BytesIO, StringIO
from pathlib import Path
import time
import urllib3
import re

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

## 1. Lectura de datos de Secretaria de Salud

In [24]:
def lectura_excel(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
    }

    # Solicitud GET para descargar el archivo
    response = requests.get(url, headers=headers)

    # Verificación de descarga
    if response.status_code == 200:
        df = pd.read_excel(BytesIO(response.content))
        print("Datos cargados.")
        return df
    else:
        print(f"Error al descargar: {response.status_code}")

In [25]:
path_2022 = 'https://datos.sonora.gob.mx/dataset/5494daa1-09ca-4750-aeac-2558553700e1/resource/4121cf57-1e4a-4fb0-b175-3805edffca2c/download/fiebre-manchada-2022.xlsx'
path_2023 = 'https://datos.sonora.gob.mx/dataset/5494daa1-09ca-4750-aeac-2558553700e1/resource/30b18862-f126-41ac-b41f-ce7126f3ac9b/download/fiebre-manchada-2023.xlsx'
path_2024 = 'https://datos.sonora.gob.mx/dataset/5494daa1-09ca-4750-aeac-2558553700e1/resource/cc6c566f-3088-41cd-b99f-169c5cf6090c/download/fiebre-manchada-2024.xlsx'

rick_2022 = lectura_excel(path_2022)
print(f"Tamaño del DataFrame 2022: {rick_2022.shape}")
rick_2023 = lectura_excel(path_2023)
print(f"Tamaño del DataFrame 2023: {rick_2023.shape}")
rick_2024 = lectura_excel(path_2024)
print(f"Tamaño del DataFrame 2024: {rick_2024.shape}")

Datos cargados.
Tamaño del DataFrame 2022: (764, 173)
Datos cargados.
Tamaño del DataFrame 2023: (873, 173)
Datos cargados.
Tamaño del DataFrame 2024: (994, 173)


### **1.1 Normalización de fechas**

In [26]:
def normalizar_fechas(df):
    cols_fecha = [col for col in df.columns if col.startswith(('fec_', 'fecha'))]
    
    for col in cols_fecha:
        df[col] = pd.to_datetime(
            df[col].astype(str).str.split(' ').str[0],
            errors='coerce'
        )
    
    return df

In [27]:
rick_2022 = normalizar_fechas(rick_2022)
rick_2023 = normalizar_fechas(rick_2023)
rick_2024 = normalizar_fechas(rick_2024)

In [28]:
rick = pd.concat([rick_2022, rick_2023, rick_2024], ignore_index=True)
rick.head()

,ide_id,ide_sex,ide_eda_ano,des_cal,ide_col,des_jur_res,des_mpo_res,cve_loc_res,des_loc_res,ide_cp,...,ricket_cq,res_final_ricket_inmuno,res_final_ricket_rtpcr,fecha_resultado_rickett_rtpcr,ricket_especie_rtpcr,tratamiento_rickettsiosis,fec_ini_trat_ricket,fec_fin_trat_ricket,med_trat_ricket,des_ins_uni_trat_norm
0,928424,1,39,CALLE,PASEOS DEL PEDREGAL Fraccionamiento,HERMOSILLO,HERMOSILLO,289,HERMOSILLO,83118.0,...,0.0,0.0,1.0,2022-01-05,0.0,1,2022-01-01,2022-01-04,1,SSA
1,928745,2,8,CALLE,SAN RAFAEL Colonia,SAN LUIS RIO COLORADO,PUERTO PEÑASCO,1,PUERTO PEÑASCO,83557.0,...,0.0,0.0,2.0,2022-01-13,0.0,1,2022-01-05,NaT,1,NaN
2,928768,2,2,CALLE,EL BOSQUE (CALLE QUINCE) Ejido,CAJEME,CAJEME,365,QUETCHEHUECA,85207.0,...,NaN,NaN,NaN,NaT,NaN,1,2022-01-06,NaT,0,SSA
3,929029,1,9,PRIVADA,PUERTO PEÑASCO CENTRO Colonia,SAN LUIS RIO COLORADO,PUERTO PEÑASCO,1,PUERTO PEÑASCO,83550.0,...,0.0,0.0,2.0,2022-01-13,0.0,1,2022-01-10,NaT,1,NaN
4,929184,2,13,CALLE,PALO BLANCO Congregación,NAVOJOA,NAVOJOA,124,TESIA,85235.0,...,0.0,1.0,1.0,NaT,0.0,1,2022-01-14,NaT,1,IMSS Ordinario


## 2. Lectura de datos de CONAGUA

In [18]:
PATRON_URL = 'https://smn.conagua.gob.mx/tools/RESOURCES/Normales_Climatologicas/Diarios/son/dia{clave}.txt'

# Rango de claves a intentar (Sonora = 26XXX)
# Se descartan automáticamente las que devuelvan error 404
CLAVE_INICIO = 26001
CLAVE_FIN    = 26405

# Carpeta donde se guardan los .txt descargados
CARPETA = Path('../data/raw/datos_sonora_diario')
CARPETA.mkdir(exist_ok=True)

PAUSA   = 0.3   # segundos entre peticiones
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

print(f'Carpeta de salida : {CARPETA.resolve()}')

Carpeta de salida : C:\Users\fanny\OneDrive\Documentos\_MCD\machine_learning\rickettsia_sonora\data\raw\datos_sonora_diario


In [ ]:
descargados = []   # claves descargadas exitosamente
omitidos    = []   # ya existían en disco
no_existen  = []   # 404 – la estación no tiene archivo Diario
errores     = []   # otros errores de red

claves = range(CLAVE_INICIO, CLAVE_FIN + 1)
total  = len(claves)

for i, clave in enumerate(claves, 1):
    nombre_archivo = f'dia{clave}.txt'
    ruta_local     = CARPETA / nombre_archivo
    url            = PATRON_URL.format(clave=clave)

    # Si ya existe en disco, saltar
    if ruta_local.exists():
        omitidos.append(clave)
        continue

    try:
        r = requests.get(url, headers=HEADERS, verify=False, timeout=20)

        if r.status_code == 200:
            ruta_local.write_bytes(r.content)
            descargados.append(clave)
        elif r.status_code == 404:
            no_existen.append(clave)   # estación sin archivo, normal
        else:
            errores.append({'clave': clave, 'status': r.status_code})

    except requests.RequestException as e:
        errores.append({'clave': clave, 'status': str(e)})

    time.sleep(PAUSA)
    
    if i % 50 == 0 or i == total:
        print(f'  {i}/{total} — descargados: {len(descargados)}, sin archivo: {len(no_existen)}, errores: {len(errores)}')

print(f'\nDescargados  : {len(descargados)}')
print(f'Ya existían  : {len(omitidos)}')
print(f'Sin archivo  : {len(no_existen)}')
print(f'Errores      : {len(errores)}')
if errores:
    print('\nDetalle de errores:')
    for e in errores:
        print(f"  clave {e['clave']}: {e['status']}")

  350/405 — descargados: 10, sin archivo: 0, errores: 64
  400/405 — descargados: 10, sin archivo: 0, errores: 114

 Descargados  : 10
Ya existían  : 279
Sin archivo  : 0
Errores      : 116

Detalle de errores:
  clave 26003: 500
  clave 26039: 500
  clave 26065: 500
  clave 26120: 500
  clave 26124: 500
  clave 26137: 500
  clave 26143: 500
  clave 26144: 500
  clave 26146: 500
  clave 26147: 500
  clave 26148: 500
  clave 26149: 500
  clave 26167: 500
  clave 26168: 500
  clave 26169: 500
  clave 26170: 500
  clave 26171: 500
  clave 26172: 500
  clave 26187: 500
  clave 26191: 500
  clave 26203: 500
  clave 26218: 500
  clave 26219: 500
  clave 26220: 500
  clave 26221: 500
  clave 26225: 500
  clave 26226: 500
  clave 26253: 500
  clave 26278: 500
  clave 26280: 500
  clave 26311: 500
  clave 26317: 500
  clave 26318: 500
  clave 26319: 500
  clave 26320: 500
  clave 26321: 500
  clave 26322: 500
  clave 26323: 500
  clave 26324: 500
  clave 26325: 500
  clave 26326: 500
  clave 26

In [20]:
# Patrones para extraer metadatos del encabezado
PATRONES_META = {
    'clave'    : r'ESTACI[OÓ]N\s*:\s*(\S+)',
    'nombre'   : r'NOMBRE\s*:\s*(.+)',
    'estado'   : r'ESTADO\s*:\s*(.+)',
    'municipio': r'MUNICIPIO\s*:\s*(.+)',
    'situacion': r'SITUACI[OÓ\xd3]N\s*:\s*(.+)',
    'latitud'  : r'LATITUD\s*:\s*([\-\d\.]+)',
    'longitud' : r'LONGITUD\s*:\s*([\-\d\.]+)',
    'altitud'  : r'ALTITUD\s*:\s*([\-\d\.]+)',
}


def parsear_estacion(ruta: Path) -> pd.DataFrame | None:
    """
    Lee un TXT del SMN y devuelve un DataFrame con columnas:
    clave, nombre, estado, municipio, situacion,
    latitud, longitud, altitud, fecha, precip, evap, tmax, tmin
    """
    try:
        contenido = ruta.read_text(encoding='utf-8')
        lineas    = contenido.splitlines()

        #Extraer metadatos
        meta = {k: None for k in PATRONES_META}
        meta['clave'] = ruta.stem.replace('dia', '')   # fallback desde nombre de archivo

        datos_inicio = None
        for i, linea in enumerate(lineas):
            # Buscar metadatos
            for campo, patron in PATRONES_META.items():
                m = re.search(patron, linea, re.IGNORECASE)
                if m:
                    meta[campo] = m.group(1).strip()

            # Detectar primera línea de datos: empieza con YYYY-MM-DD
            if re.match(r'^\d{4}-\d{2}-\d{2}', linea.strip()):
                datos_inicio = i
                break

        if datos_inicio is None:
            return None   # archivo sin datos

        # Convertir lat/lon/alt a float
        for campo in ('latitud', 'longitud', 'altitud'):
            try:
                meta[campo] = float(meta[campo]) if meta[campo] else None
            except (ValueError, TypeError):
                meta[campo] = None

        #Leer bloque de datos
        # El archivo tiene dos líneas de cabecera antes de los datos:
        #   FECHA  PRECIP  EVAP  TMAX  TMIN
        #          (mm)    (mm)  (°C)  (°C)    ← esta línea la saltamos
        # Para evitar problemas, leemos solo las líneas que empiecen con fecha
        lineas_datos = [
            l for l in lineas[datos_inicio:]
            if re.match(r'^\d{4}-\d{2}-\d{2}', l.strip())
        ]

        df = pd.read_csv(
            StringIO('\n'.join(lineas_datos)),
            sep       = r'\t',
            header    = None,
            names     = ['fecha', 'precip', 'evap', 'tmax', 'tmin'],
            engine    = 'python',
            na_values = ['NULO', 'nulo', '', ' ', '-'],
        )

        # Convertir tipos
        df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
        for col in ['precip', 'evap', 'tmax', 'tmin']:
            df[col] = pd.to_numeric(df[col], errors='coerce')

        #Agregar metadatos como columnas
        for campo in ['clave', 'nombre', 'estado', 'municipio',
                      'situacion', 'latitud', 'longitud', 'altitud']:
            df.insert(df.columns.get_loc('fecha'), campo, meta[campo])

        return df

    except Exception as e:
        print(f'{ruta.name}: {e}')
        return None


# Procesar todos los archivos 
archivos = sorted(CARPETA.glob('dia*.txt'))
total    = len(archivos)
print(f'Archivos a parsear: {len(archivos)}')

dfs = []
for i, ruta in enumerate(archivos, 1):
    df = parsear_estacion(ruta)
    if df is not None:
        dfs.append(df)

    if i % 50 == 0 or i == total:
        print(f'  {i}/{total} archivos parseados...')

df_total = pd.concat(dfs, ignore_index=True)
print(f'\n DataFrame final: {df_total.shape[0]:,} filas × {df_total.shape[1]} columnas')
df_total.head(5)

Archivos a parsear: 289
  50/289 archivos parseados...
  100/289 archivos parseados...
  150/289 archivos parseados...
  200/289 archivos parseados...
  250/289 archivos parseados...
  289/289 archivos parseados...

 DataFrame final: 3,124,034 filas × 13 columnas


,clave,nombre,estado,municipio,situacion,latitud,longitud,altitud,fecha,precip,evap,tmax,tmin
0,26001,AGUA PRIETA,SONORA,AGUA PRIETA,SUSPENDIDA,31.325833,-109.548889,1210.0,1961-02-01,0.0,NaN,19.5,2.0
1,26001,AGUA PRIETA,SONORA,AGUA PRIETA,SUSPENDIDA,31.325833,-109.548889,1210.0,1961-02-02,0.0,NaN,20.0,2.5
2,26001,AGUA PRIETA,SONORA,AGUA PRIETA,SUSPENDIDA,31.325833,-109.548889,1210.0,1961-02-03,0.0,2.7,21.5,1.8
3,26001,AGUA PRIETA,SONORA,AGUA PRIETA,SUSPENDIDA,31.325833,-109.548889,1210.0,1961-02-04,0.0,3.0,16.7,1.0
4,26001,AGUA PRIETA,SONORA,AGUA PRIETA,SUSPENDIDA,31.325833,-109.548889,1210.0,1961-02-05,0.0,1.3,11.4,4.0


In [21]:
#Concatenación de datos de estaciones
df_meta = (
    df_total
    .drop_duplicates(subset='clave')
    [['clave', 'nombre', 'municipio', 'situacion', 'latitud', 'longitud', 'altitud']]
    .reset_index(drop=True)
)

print(f'Estaciones parseadas  : {len(df_meta)}')
print(f'Sin latitud           : {df_meta["latitud"].isna().sum()}')
print(f'Sin longitud          : {df_meta["longitud"].isna().sum()}')
print(f'Sin municipio         : {df_meta["municipio"].isna().sum()}')
df_meta.head()

Estaciones parseadas  : 289
Sin latitud           : 0
Sin longitud          : 0
Sin municipio         : 0


,clave,nombre,municipio,situacion,latitud,longitud,altitud
0,26001,AGUA PRIETA,AGUA PRIETA,SUSPENDIDA,31.325833,-109.548889,1210.0
1,26002,ALAMOS,ÁLAMOS,SUSPENDIDA,27.033333,-108.950000,400.0
2,26004,ARIVECHI,ARIVECHI,OPERANDO,28.929167,-109.190833,482.0
3,26005,ARIZPE,ARIZPE,OPERANDO,30.335833,-110.167500,836.0
4,26006,BACADEHUACHI,BACADÉHUACHI,OPERANDO,29.807222,-109.140000,695.0


## 3. Exportación de datos

Datos crudos

In [29]:
rick_2022.to_csv( '../data/raw/rick_2022.csv', index=False)
rick_2023.to_csv( '../data/raw/rick_2023.csv', index=False)
rick_2024.to_csv( '../data/raw/rick_2024.csv', index=False)

Datos intermedios, sin preprocesamiento

In [31]:
# Casos de Rickettsia en Sonora de 2022 a 2024
rick.to_csv( '../data/interim/rick_hist.csv', index=False)

In [ ]:
# Normales climatológicas diarias de Sonora
df_total.to_csv('../data/interim/clima_diario.csv', index=False, encoding='utf-8-sig')

# Catálogo de estaciones climatológicas
df_meta.to_csv('../data/interim/catalogo_estaciones.csv', index=False, encoding='utf-8-sig')